![Egeria Logo](https://raw.githubusercontent.com/odpi/egeria/main/assets/img/ODPi_Egeria_Logo_color.png)

### Coco Pharmaceuticals Lab
----

# Setting up a Clinical Trial to receive data from a hospital

## Introduction

Clinical trials are used to test that new treatments are both safe and effective.  They involve taking measurements from various patients both before and after they start the treatment.

The data comes in from a variety of hospitals.  It is personally sensitive to the patients, of importance to the business, and subject to regulatory control and so care is needed that:

* it has been collected correctly
* it is protected at all times
* the correct data sharing agreements are in place to provide legal cover, both for the hospitals and Coco Pharmaceuticals.

We look at this process from the perspective of

1. The **business (clinical research)** who manage the clinical trial.  Tessa Tube is Coco Pharmaceutical's Chief Research Officer. She commissions any new trial, sets out the requirements and objectives for the study.  Tanya Tidie is responsible for the administration of the clinical trial.  She organizaes the data sharing agreements with the hospitals and monitors the progress to ensure all of the evidence is in place to show to the regulators.
   
2. The **data engineer** who sets up the data pipelines. Peter Profile is responsible for selecting the Unity Catalog that will protect the data from the clinical trial, setting up a schema and volume for the weekly measurements data in that catalog and, as the hospitals complete their registration into the clinical trail, activate the landing area and onboarding pipeline for data fro mthe hospital.

3. The **data scientist** who analyses the resulting data.  Callie Quartile is the data scientist who has to locate, access and analyse the data.  She needs to be sure the data is of sufficient quality and she has the right permissions to perform the type of analysis she wants.
   
<figure style="margin-left: 0%; display:inline-block;">
<img src="https://raw.githubusercontent.com/odpi/egeria-docs/main/site/docs/practices/coco-pharmaceuticals/personas/tessa-tube.png">
<figcaption style="margin-left: 15%;"><strong>Tessa Tube</strong></figcaption>
</figure>
    
<figure style="margin-left: 7%; display:inline-block;">
<img src="https://raw.githubusercontent.com/odpi/egeria-docs/main/site/docs/practices/coco-pharmaceuticals/personas/tanya-tidie.png">
<figcaption style="margin-left: 15%;"><strong>Tanya Tidie</strong></figcaption>
</figure>

<figure style="margin-left: 7%; display:inline-block;">
<img src="https://raw.githubusercontent.com/odpi/egeria-docs/main/site/docs/practices/coco-pharmaceuticals/personas/peter-profile.png">
<figcaption style="margin-left: 15%;"><strong>Peter Profile</strong></figcaption>
</figure>

<figure style="margin-left: 7%; display:inline-block;">
<img src="https://raw.githubusercontent.com/odpi/egeria-docs/main/site/docs/practices/coco-pharmaceuticals/personas/callie-quartile.png">
<figcaption style="margin-left: 15%;"><strong>Callie Quartile</strong></figcaption>
</figure>


----
## Checking environment set up

Lets get started by initializing the python interface to Egeria - pyegeria.

----

In [1]:
# Initialize pyegeria

import os
view_server = os.environ.get("EGERIA_VIEW_SERVER","qs-view-server")
url = os.environ.get("EGERIA_VIEW_SERVER_URL","https://host.docker.internal:9443")
user_id = "peterprofile"
user_pwd = os.environ.get("EGERIA_USER_PASSWORD")
egeria_width = 150

print("\n")
print("Accessing view server " + view_server + " on platform " + url + " for user " + user_id)
print("\n")

# These packages support the different types of markdown display
from IPython.display import display, Markdown
from pyegeria import load_mermaid, render_mermaid
load_mermaid()

# These are monitor functions from pyegeria
from commands.ops.monitor_daemon_status import display_integration_daemon_status
from commands.ops.monitor_engine_activity import display_engine_activity_c
from commands.ops.monitor_engine_status import display_gov_eng_status

import asyncio
import nest_asyncio
#nest_asyncio.apply()

from datetime import datetime
import json
import time


# EgeriaTech combines many of the clients to call Egeria
from pyegeria import EgeriaTech

egeria_client = EgeriaTech(view_server, url, user_id, user_pwd)
token = egeria_client.create_egeria_bearer_token()



Accessing view server qs-view-server on platform https://host.docker.internal:9443 for user peterprofile




----

Check that these governance engines that support the clinical trial are running ... 

----

In [2]:

display_gov_eng_status(['AssetOnboarding','AssetQuality@CocoPharmaceuticals','ClinicalTrials@CocoPharmaceuticals','UnityCatalogGovernance','UnityCatalogSurvey'],
                       status_filter=["*"],
                       engine_host = 'Coco Pharmaceuticals.qs-engine-host',  view_server = 'qs-view-server',
                       paging = True, jupyter = True,width = 150,sort = True)


                                                 Governance Engine Status @ Wed Jul  1 09:01:09 2026                                                  
╭───────────────────────────────────┬────────────────────────┬───────────────────────────────────┬───────────────┬───────────────────────────────────╮
│ Gov Engine                        │ Gov Engine Type        │ Gov Engine Desc                   │ Engine Status │ Request Types                     │
├───────────────────────────────────┼────────────────────────┼───────────────────────────────────┼───────────────┼───────────────────────────────────┤
│ AssetOnboarding                   │ GovernanceActionEngine │ Monitors, validates and enriches  │ RUNNING       │                                   │
│                                   │                        │ metadata relating to assets as    │               │  • catalog-software-server        │
│                                   │                        │ they are catalogued.           

----

A PostgreSQL database is used by the data scientist as part of their analysis of the data from the clinical trial.

----

In [3]:
# Set up the details of the PostgreSQL Server - check localhost is correct wrt where the OMAG Server Platform is running.
## Use "localhost" if the OMAG Server Platform is runing natively; use "host.docker.internal" if the OMAG Server Plaform is running in docker (default)

postgres_host_identifier="host.docker.internal"

postgres_port_number="5442"
postgres_database_name="clinical_trials"

postgres_server_name="LocalPostgreSQL1"
postgres_secrets_collection_name = "PostgreSQL Server Secret"
secrets_store_path_name = "loading-bay/secrets/integration.omsecrets"


----

Data is received into the Coco Data Lake.  This is protected by multiple deployments of Unity Catalog Server one at each site with different types of storage attached.

A Unity Catalog Server can contain multiple catalogs.  The code below assumes that you have a Unity Catalog Server running that is catalogued as `Unity Catalog 1` and there is a catalog within it called `clinical_trials`.  The **Unity Catalog Survey and Catalog Workbook** will set this up.  You can change the names if you like.

----

In [4]:
# Set up the details of the Unity Catalog Server - check localhost is correct wrt where the OMAG Server PLatform is runnng.
## Use "http://localhost:8087" if the OMAG Server Platform is runing natively; use "http://host.docker.internal:8087" if the OMAG Server Plaform is running in docker (default)
uc_server_url="http://host.docker.internal:8087"

# Catalog name used by all clinical trials - set up in AddCocoMetadataToQuickStartServers.http
catalog_name="clinical_trials"

# If all is well, a guid should be returned for the catalog, indicating that 
catalog_guid=egeria_client.get_element_guid_by_unique_name("Unity Catalog Catalog::" + uc_server_url + "::" + catalog_name)
print(catalog_guid)

f0292846-559b-4bc1-8c19-f8bcbabf15e0


-----

The next set of variables allow this notebook to run in different environments.  If you are using the standard quickstart environment then there is no need to change these values.

-----

In [5]:

tessas_user_id = "tessatube"
tessas_view_server=view_server
tessas_view_server_url=url
tessas_user_pwd = os.environ.get("EGERIA_USER_PASSWORD")

tanyas_user_id = "tanyatidie"
tanyas_view_server=view_server
tanyas_view_server_url=url
tanyas_user_pwd = os.environ.get("EGERIA_USER_PASSWORD")

callies_user_id = "calliequartile"
callies_view_server=view_server
callies_view_server_url=url
callies_user_pwd = os.environ.get("EGERIA_USER_PASSWORD")

peters_user_id="peterprofile"
peters_view_server=view_server
peters_view_server_url=url
peters_user_pwd = os.environ.get("EGERIA_USER_PASSWORD")



----

## Clinical Trial Solution Blueprint

This query shows the solution blueprint for all clinical trial processing at Coco Pharmaceuticals.  

Solution blueprint definitions are created through a simple markdown document that is loaded into Egeria using the Dr.Egeria feature.  It is a generic description of how a particular type of project (clinical trials in this example) operate.  A solution blueprint contains a description of the people (or more specifically the roles of the people) involved in this type of project along with both the manual and automated processes in use. It provides a convenient structure for linking descriptions of the governance requirements and responsibilities into a business operational view.  Thus the solution blueprint consolidates the aspects of an activity that the organisation wishes to be consistent.  This may be for cost, or quality and/or regulatory reasons.
The solution blueprint also provides a familiar structure to to report the operational status of each part of the end-to-end processing

Now look at this solution blueprint in detail.  

If we start from the top, there is data coming in from one or more hospitals, and each hospital has a coordinator that ensures the hospital is delivering the data as agreed.  We don’t really know exactly how each hospital manages their data - just that whatever processes they have, will deliver the appropriate data into the landing area folder for the hospital.  

The delivery of data from a hospital is immediately detected by the landing folder cataloguer which initiates an onboarding request for the data.  

The onboarding pipeline is responsible for moving the data to the safety of the data lake and ensuring it is correctly catalogued.  This is a critical step, and it is set up and managed by a data engineer, who is supported by the clinical trial manager if there are issues with the data from any of the hospitals.  Once  the data is in the data lake, it is picked up by the clinical trials researchers and data scientists for analysis and ultimately to deliver a report summarising the findings of the clinical trial.

Notice that down the centre of the solution blueprint is a high level view of how the data for the clinical trial flows.   The processes and roles that support this data flow are linked to either side.


In [6]:
token = egeria_client.create_egeria_bearer_token()

solution_blueprint=egeria_client.get_element_by_unique_name("SolutionBlueprint::Clinical Trial Management::Clinical Trial Management Solution Blueprint")
render_mermaid(solution_blueprint.get("solutionBlueprintMermaidGraph"))


----

Since the solution blueprint is generic, projects are used to track the progress of individual clinical trials.  These are created using templates associated with the solution blueprint.  These templates help to ensure consistency from project to project.   One of the key templates is the information suppy chain template.  This identifies which components in the solution blueprint are responsible for lineage.  It also divides the data flow into segments.  A segment shows the part of the data flow that is the responsibility of a particular team, organization or persion.  For clinical trials, this includes:

* The delivery of data from the hospitals
* The onboarding and validating of data from the hospitals into the data lake
* The provisioning of data into the sandboxes used for analysing the clinical trial measurements
* The assessment of the clinical trial data
* The reporting of the results


Here is the information supply chain template for clinical trials ... notice that some solution components are linked to multiple solution components.  This signifies a handover of responsibility for the data.

----

In [7]:
token = egeria_client.create_egeria_bearer_token()

information_supply_chain_guid=egeria_client.get_element_guid_by_unique_name("InformationSupplyChain::Clinical Trials Information Supply Chain")
if information_supply_chain_guid != "No elements found":
    information_supply_chain=egeria_client.get_info_supply_chain_by_guid(guid=information_supply_chain_guid)
    render_mermaid(information_supply_chain.get("iscimplementationMermaidGraph"))
else:
    print(information_supply_chain_guid)


In [8]:
token = egeria_client.create_egeria_bearer_token()

information_supply_chain_guid=egeria_client.get_element_guid_by_unique_name("InformationSupplyChain::Clinical Trial Information Supply Chain: ~{clinicalTrialId}~")
if information_supply_chain_guid != "No elements found":
    information_supply_chain=egeria_client.get_info_supply_chain_by_guid(guid=information_supply_chain_guid)
    render_mermaid(information_supply_chain.get("iscimplementationMermaidGraph"))
else:
    print(information_supply_chain_guid)



In [9]:
token = egeria_client.create_egeria_bearer_token()

information_supply_chain_guid=egeria_client.get_element_guid_by_unique_name("InformationSupplyChain::Clinical Trial Treatment Validation Information Supply Chain: ~{clinicalTrialId}~")
if information_supply_chain_guid != "No elements found":
    information_supply_chain=egeria_client.get_info_supply_chain_by_guid(guid=information_supply_chain_guid)
    render_mermaid(information_supply_chain.get("iscimplementationMermaidGraph"))
else:
    print(information_supply_chain_guid)


-----

Other [templates](https://egeria-project.org/features/templated-cataloguing/overview/) are used to ensure that as digital resources are catalogued, they have the right structure, classifications and attachments.

----

In [10]:
token = egeria_client.create_egeria_bearer_token()

# Identify the standard onboarding pipeline

onboarding_pipeline_name = "Onboard Landing Area Files For Clinical Trial Project"
onboarding_pipeline_guid = egeria_client.get_guid_for_name(onboarding_pipeline_name) 
print("onboardingPipeline: " + onboarding_pipeline_guid)


# Identify the templates that are used to create consistency in the clinical trial process

data_lake_schema_template_guid=egeria_client.get_element_guid_by_unique_name("Unity Catalog Schema::~{serverNetworkAddress}~::~{ucCatalogName}~.~{ucSchemaName}~")
print("dataLakeSchemaTemplateGUID: " + data_lake_schema_template_guid)

data_lake_volume_template_guid=egeria_client.get_element_guid_by_unique_name("Unity Catalog Volume::~{serverNetworkAddress}~::~{ucCatalogName}~.~{ucSchemaName}~.~{ucVolumeName}~")
print("dataLakeVolumeTemplateGUID: " + data_lake_volume_template_guid)

data_lake_file_template_guid = egeria_client.get_element_guid_by_unique_name("DataLake::~{clinicalTrialId}~::WeeklyMeasurements::~{hospitalName}~::~{fileName}~")
print("dataLakeFileTemplateGUID: " + data_lake_file_template_guid)

landing_area_directory_template_guid = egeria_client.get_element_guid_by_unique_name("Data Folder::~{fileSystemName}~:~{directoryPathName}~")
print("landingAreaDirectoryTemplateGUID: " + landing_area_directory_template_guid)

landing_area_file_template_guid = egeria_client.get_element_guid_by_unique_name("LandingArea::~{hospitalName}~::~{clinicalTrialId}~::WeeklyMeasurements::~{fileName}~")
print("landingAreaFileTemplateGUID: " + landing_area_file_template_guid)

validated_weekly_files_template_guid = egeria_client.get_element_guid_by_unique_name("Data File Collection::~{displayName}~")
print("validatedWeeklyFilesTemplateGUID: " + validated_weekly_files_template_guid)


onboardingPipeline: b16eaf01-bae3-40fe-888a-385a46c7e663
dataLakeSchemaTemplateGUID: 5bf92b0f-3970-41ea-b0a3-aacfbf6fd92e
dataLakeVolumeTemplateGUID: 92d2d2dc-0798-41f0-9512-b10548d312b7
dataLakeFileTemplateGUID: 4ef24563-1df0-4d2a-8004-a7f187797b62
landingAreaDirectoryTemplateGUID: 372a0379-7060-4c9d-8d84-bc709b31794c
landingAreaFileTemplateGUID: 52607215-da3e-4f78-87e1-20ddc7eb80bb
validatedWeeklyFilesTemplateGUID: 26d6bcdc-ce05-4e0b-8685-cd40777dc5f9


---

At Coco Pharmaceuticals, clinical trial projects are part of the *Clinical Trial Management* campaign.  This is a project that is responsible for the generic resources used by all clinical trials.

---

In [11]:
token = egeria_client.create_egeria_bearer_token()

parent_project_name="Campaign::Clinical Trials Management"
parent_project_guid=egeria_client.get_element_guid_by_unique_name(parent_project_name)

print("Parent project: " + parent_project_name)
print("  parent project GUID: " + parent_project_guid)

print(parent_project_guid)
parent_project=egeria_client.get_project_by_guid(project_guid=parent_project_guid)
render_mermaid(parent_project.get("mermaidGraph"))


Parent project: Campaign::Clinical Trials Management
  parent project GUID: ef7b89d1-96f9-4892-b9ea-fab0ee0121e0
ef7b89d1-96f9-4892-b9ea-fab0ee0121e0


----

The definitions and functions that follow are generic for all clinical trials that are set up in this notebook.  They are added here to simplify the steps in each clinical trial.

----

In [12]:
# These functions drive calls to Egeria at different phases of the clinical trial


def set_up_clinical_trial(egeria_tech,parent_project_guid, clinical_trial_owner_guid, clinical_trial_manager_guid, project_manager_guid, data_engineer_guid, integration_developer_guid, data_scientist_guid, project_identifier, project_name, project_description): 
    set_up_clinical_trial_name="ClinicalTrials@CocoPharmaceuticals::set-up-clinical-trial"
    action_targets = [{
          "class" : "NewActionTarget",
          "actionTargetName": "clinicalTrialParentProject",
          "actionTargetGUID": parent_project_guid
        },
        {
          "class" : "NewActionTarget",
          "actionTargetName": "clinicalTrialOwner",
          "actionTargetGUID": clinical_trial_owner_guid 
        },
        {
          "class" : "NewActionTarget",
          "actionTargetName": "clinicalTrialManager",
          "actionTargetGUID": clinical_trial_manager_guid 
        },
        {
          "class" : "NewActionTarget",
          "actionTargetName": "itProjectManager",
          "actionTargetGUID": project_manager_guid
        },
        {
          "class" : "NewActionTarget",
          "actionTargetName": "dataEngineer",
          "actionTargetGUID": data_engineer_guid
        },
        {
          "class" : "NewActionTarget",
          "actionTargetName": "integrationDeveloper",
          "actionTargetGUID": integration_developer_guid
        },
        {
          "class" : "NewActionTarget",
          "actionTargetName": "dataScientist",
          "actionTargetGUID": data_scientist_guid
        }]
    
    request_parameters = request_parameters = {
         "clinicalTrialId" : project_identifier,
         "clinicalTrialName" : project_name,
         "clinicalTrialDescription" : project_description
    }
    
    return egeria_tech.initiate_gov_action_type(set_up_clinical_trial_name, None, action_targets, None, request_parameters, None, None)


def nominate_hospital(egeria_tech, project_identifier, hospital_guid, contact_person_guid):
    nominate_hospital_name = "ClinicalTrials::" + project_identifier + "::nominate-hospital"
    process_guid = egeria_tech.get_element_guid_by_unique_name(nominate_hospital_name)
    print(nominate_hospital_name + " [" + process_guid + "] running for hospital: " + hospital_guid)
    if process_guid != "No elements found":
        action_targets = [{
              "class" : "NewActionTarget",
              "actionTargetName": "hospital",
              "actionTargetGUID": hospital_guid
            },
            {
              "class" : "NewActionTarget",
              "actionTargetName": "hospitalContactPerson",
              "actionTargetGUID": contact_person_guid
            }]
        
        return egeria_tech.initiate_gov_action_process(nominate_hospital_name, None, action_targets, datetime.now(), None, None, None)
    else:
        return None



def set_up_data_lake(egeria_tech, project_identifier, project_name, project_directory_name, project_schema_name):
    set_up_data_lake_process_name="ClinicalTrials::" + project_identifier + "::set-up-data-lake"
    process_guid = egeria_tech.get_element_guid_by_unique_name(set_up_data_lake_process_name)
    print(set_up_data_lake_process_name + " [" + process_guid + "] running")
    if process_guid != "No elements found":    
        data_lake_directory_path_name="coco-data-lake/research/clinical-trials/" + project_directory_name + "/weekly-measurements"
        
        airflow_dag_name="populate_" + project_schema_name + "_sandbox"
        
        action_targets = [{
              "class" : "NewActionTarget",
              "actionTargetName": "dataLakeCatalog",
              "actionTargetGUID": catalog_guid
            },
            {
              "class" : "NewActionTarget",
              "actionTargetName": "onboardingPipeline",
              "actionTargetGUID": onboarding_pipeline_guid 
            }]
        
        request_parameters = {
             "dataLakeVolumeTemplateGUID" : data_lake_volume_template_guid,
             "dataLakeSchemaTemplateGUID" : data_lake_schema_template_guid,
             "dataLakeFileTemplateGUID" : data_lake_file_template_guid,
             "landingAreaDirectoryTemplateGUID" : landing_area_directory_template_guid,
             "landingAreaFileTemplateGUID" : landing_area_file_template_guid,
             "dataLakeSchemaName" : project_schema_name,
             "dataLakeSchemaDescription" : "Data for the " + project_name + ".",
             "dataLakeVolumeName" : "weekly_measurements",
             "dataLakeVolumeDescription" : "Weekly patient measurements",
             "dataLakeVolumeDirectoryPathName" : data_lake_directory_path_name,
             "validatedWeeklyFilesDataSetName" : "Validated Incoming Weekly Measurements for the " + project_name + ".",
             "validatedWeeklyFilesTemplateGUID" : validated_weekly_files_template_guid,
             "airflowDAGName" : airflow_dag_name,
             "serverName" : postgres_server_name,
             "hostIdentifier" : postgres_host_identifier,
             "portNumber" : postgres_port_number,
             "secretsStorePathName" : secrets_store_path_name,
             "secretsCollectionName" : postgres_secrets_collection_name,
             "versionIdentifier" : "1.0",
             "databaseName" : postgres_database_name,
             "schemaName" : project_schema_name,
             "schemaDescription" : "PostgreSQL database schema for the " + project_name + "."
        }
        
        egeria_tech.initiate_gov_action_process(set_up_data_lake_process_name, None, action_targets, None, request_parameters, None, None)
        return airflow_dag_name
    else:
        return None



def certify_hospital(egeria_tech, project_identifier, hospital_guid):
    certify_hospital_name = "ClinicalTrials::" + project_identifier + "::certify-hospital"
    process_guid = egeria_tech.get_element_guid_by_unique_name(certify_hospital_name)
    print(certify_hospital_name + " [" + process_guid + "] running for hospital: " + hospital_guid)
    if process_guid != "No elements found":
        action_targets = [{
              "class" : "NewActionTarget",
              "actionTargetName": "hospital",
              "actionTargetGUID": hospital_guid
            }]
        return egeria_tech.initiate_gov_action_process(certify_hospital_name, None, action_targets, datetime.now(), None, None, None)
    else:
        return None



def onboard_hospital(egeria_tech, project_identifier, project_directory_name, hospital_guid, hospital_landing_area_folder):
    onboard_hospital_name = "ClinicalTrials::" + project_identifier + "::onboard-hospital"
    process_guid = egeria_tech.get_element_guid_by_unique_name(onboard_hospital_name)
    print(onboard_hospital_name + " [" + process_guid + "] running for hospital: " + hospital_guid)
    if process_guid != "No elements found":
        landing_area_directory_name="landing-area/hospitals/" + hospital_landing_area_folder + "/clinical-trials/" + project_directory_name
        actionTargets = [{
              "class" : "NewActionTarget",
              "actionTargetName": "hospital",
              "actionTargetGUID": hospital_guid
            }]
        requestParameters = {
            "landingAreaDirectoryPathName" : landing_area_directory_name
        }
        egeria_tech.initiate_gov_action_process(onboard_hospital_name, None, actionTargets, datetime.now(), requestParameters, None, None)
        return landing_area_directory_name
    else:
        return None
        

In [13]:
# These are the sample data files found in Egeria's distribution

oak_dene_source_folder   = 'loading-bay/sample-data/oak-dene-drop-foot-weekly-measurements'
old_market_source_folder = 'loading-bay/sample-data/old-market-drop-foot-weekly-measurements'
hampton_source_folder    = 'loading-bay/sample-data/hampton-drop-foot-weekly-measurements'

# This function simulates the delivery of a file from a hospital.

def add_file_to_landing_area(egeria_client, source_folder, destination_folder, week_number):
    add_file_to_landing_area_name="ClinicalTrials@CocoPharmaceuticals::simulate-ftp"
    source_file_name = source_folder + "/" + "week" + week_number + ".csv"
    print("Moving " + source_file_name + " to " + destination_folder)
    request_parameters = {
        "sourceFile" : source_file_name,
        "destinationDirectory" : destination_folder
    }
    egeria_client.initiate_gov_action_type(add_file_to_landing_area_name, None, None, None, request_parameters, None)
    

----

## Teddy Bear Drop Foot Clinical Trial

In this first clinical trial, we step through each phase slowly and observe the behaviour of the associated templates and governance actions.  

### Initializing the clinical trial

Tessa Tube is the lead researcher at Coco Pharmaceuticals.  She is the *Clinical Trial Sponsor*.   In addition to monitoring the clinical trial throughout its lifetime, she has two specific actions to perform, one at the start of the trial and one at the end:


---

In [14]:

tessas_client = EgeriaTech(tessas_view_server, tessas_view_server_url, tessas_user_id, tessas_user_pwd)
token = tessas_client.create_egeria_bearer_token()

solution_role_guid=tessas_client.get_element_guid_by_unique_name("SolutionActorRole::Coco::ClinicalTrialSponsor")
print(solution_role_guid)
solution_role=tessas_client.get_solution_role_by_guid(guid=solution_role_guid)
render_mermaid(solution_role.get("mermaidGraph"))


f6bc847b-868d-43cc-b767-41f5fe3e47d1


----

The clinical trial that Tessa is setting up is for the `Teddy Bear Drop Foot` condition that afflicts elderly teddy bears as their stuffung sags, loosening their hip joints.  The clinical trial is given an identifier, name and description along with resource names used to set up specific file directories and schema for the data being processed by each stage in the clinicl trial.

----

In [15]:
print()

project_identifier="PROJ-CT-TBDF"
project_name="Teddy Bear Drop Foot Clinical Trial"
project_description="Clinical trial related to the new treatment for Teddy Bear Drop Foot."
project_directory_name="teddy-bear-drop-foot"
project_schema_name="teddy_bear_drop_foot"

print("Project " + project_identifier + ": " + project_name)
print("  " + project_description)

print()



Project PROJ-CT-TBDF: Teddy Bear Drop Foot Clinical Trial
  Clinical trial related to the new treatment for Teddy Bear Drop Foot.



----

The people involved in the clinical trial are identified.  There is a person for each of the solution roles in the blueprint.

----

In [16]:
print()

clinical_trial_owner_name="Tessa Tube"
print("clinical trial owner name: " + clinical_trial_owner_name)
clinical_trial_owner_guid = tessas_client.get_guid_for_name(clinical_trial_owner_name)
print("clinical trial owner GUID: " + clinical_trial_owner_guid)

print()
clinical_trial_manager_name="Tanya Tidie"
print("clinical trial manager name: " + clinical_trial_manager_name)
clinical_trial_manager_guid = tessas_client.get_guid_for_name(clinical_trial_manager_name)
print("clinical trial manager GUID: " + clinical_trial_manager_guid)

print()
data_scientist_name="Callie Quartile"
print("data scientist name: " + data_scientist_name)
data_scientist_guid = tessas_client.get_guid_for_name(data_scientist_name)
print("data scientist GUID: " + data_scientist_guid)

print()
data_engineer_name="Peter Profile"
print("data engineer name: " + data_engineer_name)
data_engineer_guid = tessas_client.get_guid_for_name(data_engineer_name) 
print("data engineer GUID: " + data_engineer_guid)

print()
project_manager_name="Polly Tasker"
print("project manager name: " + project_manager_name)
project_manager_guid = tessas_client.get_guid_for_name(project_manager_name) 
print("project manager GUID: " + project_manager_guid)

print()
integration_developer_name="Bob Nitter"
print("integration developer name: " + integration_developer_name)
integration_developer_guid = tessas_client.get_guid_for_name(integration_developer_name) 
print("integration developer GUID: " + integration_developer_guid)

print()


clinical trial owner name: Tessa Tube
clinical trial owner GUID: f73d5c58-66c6-444c-9b55-5d7e1d1160af

clinical trial manager name: Tanya Tidie
clinical trial manager GUID: b0c856e1-bd64-4ac8-8a31-dfe94789ba91

data scientist name: Callie Quartile
data scientist GUID: fce76f01-3746-4e58-a6ac-cdb7cb362d8c

data engineer name: Peter Profile
data engineer GUID: a3c5082c-c842-4341-a2bd-421b987dabe6

project manager name: Polly Tasker
project manager GUID: ca72c491-331c-481c-b08f-bcef3a1c5e09

integration developer name: Bob Nitter
integration developer GUID: 7416a69b-5b6e-43f0-bd1e-b52013ee9225



---

Once Tessa has an agreed scope and purpose for a new clinical trial, she is ready to set up the project resources that will control the clinical trial.

---

In [17]:
token = tessas_client.create_egeria_bearer_token()

solution_component_guid=tessas_client.get_element_guid_by_unique_name("SolutionComponent::Set up clinical trial::Set-up-clinical-trial")
if solution_component_guid != "No elements found":
    solution_component=tessas_client.get_solution_component_by_guid(guid=solution_component_guid)
    render_mermaid(solution_component.get("mermaidGraph"))
else:
    print(solution_component_guid)



---

The `set-up-clinical-trial` governance action type creates the specific governance processes that will be used by the staff during the clinical trial.  It uses generic process definitions to create these specific processes that are initialized with all of the correct values for the clinical trial.  This approach reduces the chance that someone will use a wrong value by accident when running the later processes for the clinical trial.

---

In [18]:
token = tessas_client.create_egeria_bearer_token()

governance_action_guid=tessas_client.get_element_guid_by_unique_name("ClinicalTrials@CocoPharmaceuticals::set-up-clinical-trial")
print(governance_action_guid)
governance_action=tessas_client.get_governance_definition_by_guid(guid=governance_action_guid, for_lineage=False)
render_mermaid(governance_action.get("mermaidGraph"))


23ceab08-f644-49c7-b7f3-95f39fe41c84


In [19]:

set_up_action_guid = set_up_clinical_trial(tessas_client,
                                           parent_project_guid,
                                           clinical_trial_owner_guid,
                                           clinical_trial_manager_guid,
                                           project_manager_guid,
                                           data_engineer_guid,
                                           integration_developer_guid,
                                           data_scientist_guid,
                                           project_identifier, 
                                           project_name,
                                           project_description) 


----

The command below shows the status of the governance action.

----

In [20]:

token = tessas_client.create_egeria_bearer_token()

# Show the execution path of the process
governance_action=tessas_client.get_asset_by_guid(guid=set_up_action_guid)
render_mermaid(governance_action.get("mermaidGraph"))


-----

The set-up-clinical-trial governance action has created the following processes that are specific to this clinical trial and are set up with details of the correct project and people.

-----

In [21]:
token = tessas_client.create_egeria_bearer_token()

clinical_trial_processes = tessas_client.find_projects(search_string=project_identifier,
                                                       output_format="LIST",
                                                       report_spec="Projects")

display(Markdown(clinical_trial_processes))


### Projects Table

Projects found from the search string: `PROJ-CT-TBDF`

| Display Name | Qualified Name | Category | Description | Type Name | URL | Classifications | Priority | Project Status | Element Status | Start Date | Assigned Actors | Resources | Project Roles | Managed Projects | 
|-------------|-------------|-------------|-------------|-------------|-------------|-------------|-------------|-------------|-------------|-------------|-------------|-------------|-------------|-------------|
| Create Base Data Sharing Agreement for Teddy Bear Drop Foot Clinical Trial | Project::PROJ-CT-TBDF::DataSharingAgreement |  ---  | Define the data sharing agreement offered to hospitals participating in this clinical trial. |  ---  |  ---  |  ---  | 0 |  ---  |  ---  |  ---  | Person::USA::302145, Person::USA::209482 |  ---  |  ---  |  ---  | 
| Manage hospitals relationships for Teddy Bear Drop Foot Clinical Trial | Project::PROJ-CT-TBDF::HospitalManagement |  ---  | Set up the data sharing agreements and ensure certification and data delivery by the hospitals chosen to support this clinical trial. |  ---  |  ---  |  ---  | 0 |  ---  |  ---  |  ---  | Person::USA::209482 |  ---  |  ---  |  ---  | 
| Create new data quality survey services for Teddy Bear Drop Foot Clinical Trial | Project::PROJ-CT-TBDF::DataQualityComponents |  ---  | Create the survey action services used to validate the values delivered to this clinical trial. |  ---  |  ---  |  ---  | 0 |  ---  |  ---  |  ---  | Person::NL::458109 |  ---  |  ---  |  ---  | 
| Create new templates and data stores for Teddy Bear Drop Foot Clinical Trial | Project::PROJ-CT-TBDF::TemplatesAndStores |  ---  | Define the metadata templates and data store definitions to support this clinical trial. |  ---  |  ---  |  ---  | 0 |  ---  |  ---  |  ---  | Person::UK::986419 |  ---  |  ---  |  ---  | 
| Teddy Bear Drop Foot Clinical Trial | Project::PROJ-CT-TBDF::Teddy Bear Drop Foot Clinical Trial |  ---  | Clinical trial related to the new treatment for Teddy Bear Drop Foot. |  ---  |  ---  |  ---  | 0 |  ---  |  ---  |  ---  | Person::USA::302145 | DataSpec::PROJ-CT-TBDF |  ---  | Project::PROJ-CT-TBDF::DataSpecification, Project::PROJ-CT-TBDF::ITDevelopment, Project::PROJ-CT-TBDF::OnboardingPipelines, Project::PROJ-CT-TBDF::HospitalManagement, Project::PROJ-CT-TBDF::DataSharingAgreement, Project::PROJ-CT-TBDF::DataAnalysis | 
| Create new components for Teddy Bear Drop Foot Clinical Trial | Project::PROJ-CT-TBDF::ITDevelopment |  ---  | Define the data-oriented components to support this clinical trial. |  ---  |  ---  |  ---  | 0 |  ---  |  ---  |  ---  | Person::NL::338575 |  ---  |  ---  | Project::PROJ-CT-TBDF::TemplatesAndStores, Project::PROJ-CT-TBDF::DataQualityComponents | 
| Create Data Specification for Teddy Bear Drop Foot Clinical Trial | Project::PROJ-CT-TBDF::DataSpecification |  ---  | Define the specification of the data needed to conduct this clinical trial. |  ---  |  ---  |  ---  | 0 |  ---  |  ---  |  ---  | Person::USA::328080, Person::USA::302145, Person::USA::209482 | DataSpec::PROJ-CT-TBDF |  ---  |  ---  | 
| Data analysis for Teddy Bear Drop Foot Clinical Trial | Project::PROJ-CT-TBDF::DataAnalysis |  ---  | Analyse the data from the hospitals to support this clinical trial. |  ---  |  ---  |  ---  | 0 |  ---  |  ---  |  ---  | Person::USA::328080 |  ---  |  ---  |  ---  | 
| Weekly data onboarding pipelines for Teddy Bear Drop Foot Clinical Trial | Project::PROJ-CT-TBDF::OnboardingPipelines |  ---  | Set up data onboarding pipelines from hospitals to support this clinical trial. |  ---  |  ---  |  ---  | 0 |  ---  |  ---  |  ---  | Person::UK::986419 |  ---  |  ---  |  ---  | 


---

These processes are linked to the appropriate tasks in the new clinical trial project that were also defined by the set up process...

---

In [22]:
def print_project(egeria_tech, project_guid):
    element = egeria_tech.get_project_by_guid(project_guid)
    if element:
        if type(element) == str:
            print(element)
        else:
            mermaid_graph = element.get("mermaidGraph")
            render_mermaid(mermaid_graph)
            

token = tessas_client.create_egeria_bearer_token()

project_guid=tessas_client.get_element_guid_by_unique_name("Project::" + project_identifier + "::" + project_name)
print(project_guid)
if project_guid != "No elements found":
    print_project(tessas_client, project_guid)


abf3b15a-9247-4311-8557-b5819c4a5f89


----

Notice an empty data specification has been created and linked to the appropriate tasks.  The dark lines show the dependencies between  the tasks.  Completing this data specification is required before many tasks can proceed.

There is also the [information supply chain](https://egeria-project.org/concepts/information-supply-chain/) that identifies the lineage that is relevant to this project.  Notice that the project details are filled out in the placeholders of the information supply chain template.  On the left are the **segments** showing how the responsbility for the data is passed from team to team.  These are linked to the relevant solution components supporting the data processing/movement.  As the project progresses and resources are deployed to support it, these resources will appear in the implementation of the information supply chain.

---

In [23]:
token = tessas_client.create_egeria_bearer_token()

information_supply_chain_guid=tessas_client.get_element_guid_by_unique_name("InformationSupplyChain::Clinical Trial Treatment Validation Information Supply Chain: " + project_identifier)
if information_supply_chain_guid != "No elements found":
    information_supply_chain=egeria_client.get_info_supply_chain_by_guid(guid=information_supply_chain_guid)
    render_mermaid(information_supply_chain.get("iscimplementationMermaidGraph"))
else:
    print(information_supply_chain_guid)



----

### Nominating Hospitals

Tanya Tidie is the clinical trial manager, responsible for the business arrangements for each clinical trial. This includes managing the relationships with the hospitals, ensuring all of the "paperwork" associated with the clinical trial is in place for the sponsors and regulators as well as handling any issues with the data received from the hospitals.

----

In [24]:

tanyas_client = EgeriaTech(tanyas_view_server, tanyas_view_server_url, tanyas_user_id, tanyas_user_pwd)
token = tanyas_client.create_egeria_bearer_token()

solution_role=tanyas_client.get_element_by_unique_name("SolutionActorRole::Coco::ClinicalTrialManager")
if solution_role != "No elements found":
    render_mermaid(solution_role.get("mermaidGraph"))
else:
    print(solution_role)


---

These are the hospitals and associated contact people that Tanya works with.

----

In [25]:

# Identify the hospitals that take part in one or more clinical trials.
token = tanyas_client.create_egeria_bearer_token()

print()

oak_dene_hospital_name="Oak Dene Hospital"
print(oak_dene_hospital_name)
oak_dene_hospital_guid=tanyas_client.get_element_guid_by_unique_name("Organization::" + oak_dene_hospital_name)
print("  oakDeneHospitalGUID: " + oak_dene_hospital_guid)
oak_dene_contact_person_name="Robbie Records"
print("  oakDeneContactPersonName: " + oak_dene_contact_person_name)
oak_dene_contact_person_guid=tanyas_client.get_guid_for_name(oak_dene_contact_person_name)
print("  oakDeneContactPersonGUID: " + oak_dene_contact_person_guid)
oak_dene_landing_folder_directory="oak-dene"

print()
old_market_hospital_name="Old Market Hospital"
print(old_market_hospital_name)
old_market_hospital_guid=tanyas_client.get_element_guid_by_unique_name("Organization::" + old_market_hospital_name)
print("  oldMarketHospitalGUID: " + old_market_hospital_guid)
old_market_contact_person_name="Nellie Dunn"
print("  oldMarketContactPersonName: " + old_market_contact_person_name)
old_market_contact_person_guid=tanyas_client.get_guid_for_name(old_market_contact_person_name)
print("  oldMarketContactPersonGUID: " + old_market_contact_person_guid)
old_market_landing_folder_directory="old-market"


print()
hampton_hospital_name="Hampton Hospital"
print(hampton_hospital_name)
hampton_hospital_guid=tanyas_client.get_element_guid_by_unique_name("Organization::" + hampton_hospital_name)
print("  hamptonHospitalGUID: " + hampton_hospital_guid)
hampton_contact_person_name="Grant Able"
print("  hamptonContactPersonName: " + hampton_contact_person_name)
hampton_contact_person_guid=tanyas_client.get_guid_for_name(hampton_contact_person_name)
print("  hamptonContactPersonGUID: " + hampton_contact_person_guid)
hampton_landing_folder_directory="hampton"


print()
bowden_arrow_hospital_name="Bowden Arrow Hospital"
print(bowden_arrow_hospital_name)
bowden_arrow_hospital_guid=tanyas_client.get_element_guid_by_unique_name("Organization::" + bowden_arrow_hospital_name)
print("  bowdenArrowHospitalGUID: " + bowden_arrow_hospital_guid)
bowden_arrow_contact_person_name="Julie Stitched"
print("  bowdenArrowContactPersonName: " + bowden_arrow_contact_person_name)
bowden_arrow_contact_person_guid=tanyas_client.get_guid_for_name(bowden_arrow_contact_person_name)
print("  bowdenArrowContactPersonGUID: " + bowden_arrow_contact_person_guid)
bowden_arrow_landing_folder_directory="bowden-arrow"
print()
print("Completed")




Oak Dene Hospital
  oakDeneHospitalGUID: a3d40103-9e88-4dcb-9c05-82d979c87d3b
  oakDeneContactPersonName: Robbie Records
  oakDeneContactPersonGUID: 78af556d-7e8b-4063-bd36-8c17021c314c

Old Market Hospital
  oldMarketHospitalGUID: 0c899f47-d22f-4d73-86f8-4d30f676acf5
  oldMarketContactPersonName: Nellie Dunn
  oldMarketContactPersonGUID: e579602f-3a2e-4f3f-8bf1-2658803dbc62

Hampton Hospital
  hamptonHospitalGUID: 27290e7a-9108-4394-8fdd-cfcee501301a
  hamptonContactPersonName: Grant Able
  hamptonContactPersonGUID: d116451b-0abe-4f51-bf90-f9fbe0a7fabe

Bowden Arrow Hospital
  bowdenArrowHospitalGUID: 6bdd3666-7d46-4427-9316-1df0dce8d6cc
  bowdenArrowContactPersonName: Julie Stitched
  bowdenArrowContactPersonGUID: 8e2f8a1f-6c2a-4089-9599-89001ad2437e

Completed


----

Tanya's work with the hospital is tracked under the "HospitalManagement" task for this project.  This task is dependent on completing the data specification for the project sicne it plays a major role when setting up the data sharing agreements with the hospitals.

----

In [26]:
token = tanyas_client.create_egeria_bearer_token()

project=tanyas_client.get_element_by_unique_name("Project::" + project_identifier + "::HospitalManagement")
if project != "No elements found":
    render_mermaid(project.get("mermaidGraph"))
else:
    print(project)


----

Once the data specification is in place, and the basic data sharing agreement has been defined, Tanya begins to work with the hospitals that will take part in the clinical trial.  For each one, she runs nominate-hospital to identify the people at the hospital that are going to complete the requirements (training, signing data sharing agreements, ...) necessary to participate.

----

In [27]:
token = tanyas_client.create_egeria_bearer_token()

nominate_hospital_solution_component=tanyas_client.get_element_by_unique_name("SolutionComponent::Nominate Hospital::V1.0")
if nominate_hospital_solution_component != "No elements found":
    render_mermaid(nominate_hospital_solution_component.get("mermaidGraph"))
else:
    print(nominate_hospital_solution_component)
    

----

In this clinical trial, patients are associated with three hospitals:

* Oak Dene Hospital
* Old Market Hospital
* Hampton Hospital

----

In [28]:
token = tanyas_client.create_egeria_bearer_token()

# Nominate each hospital

oak_dene_process_guid=nominate_hospital(tanyas_client, project_identifier, oak_dene_hospital_guid, oak_dene_contact_person_guid)
old_market_process_guid=nominate_hospital(tanyas_client, project_identifier, old_market_hospital_guid, old_market_contact_person_guid)
hampton_hospital_process_guid=nominate_hospital(tanyas_client, project_identifier, hampton_hospital_guid, hampton_contact_person_guid)


ClinicalTrials::PROJ-CT-TBDF::nominate-hospital [ab796eae-2e0a-4e3d-aa50-7b8343b66687] running for hospital: a3d40103-9e88-4dcb-9c05-82d979c87d3b
ClinicalTrials::PROJ-CT-TBDF::nominate-hospital [ab796eae-2e0a-4e3d-aa50-7b8343b66687] running for hospital: 0c899f47-d22f-4d73-86f8-4d30f676acf5
ClinicalTrials::PROJ-CT-TBDF::nominate-hospital [ab796eae-2e0a-4e3d-aa50-7b8343b66687] running for hospital: 27290e7a-9108-4394-8fdd-cfcee501301a


In [29]:
# Check that each nominate hospital process is "COMPLETED"

token = tanyas_client.create_egeria_bearer_token()

# Show the execution path of the process
process_graph = tanyas_client.get_governance_process_graph(oak_dene_process_guid)
render_mermaid(process_graph.get("governanceActionProcessMermaidGraph"))
process_graph = tanyas_client.get_governance_process_graph(old_market_process_guid)
render_mermaid(process_graph.get("governanceActionProcessMermaidGraph"))
process_graph = tanyas_client.get_governance_process_graph(hampton_hospital_process_guid)
render_mermaid(process_graph.get("governanceActionProcessMermaidGraph"))

----

### Setting up the data lake

Peter Profile is a data engineer at Coco Pharmaceuticals and is reponsible for the data pipelines and data stores that support the clinical trial.

---

In [30]:

peters_client = EgeriaTech(peters_view_server, peters_view_server_url, peters_user_id, peters_user_pwd)
token = peters_client.create_egeria_bearer_token()

solution_role=peters_client.get_element_by_unique_name("SolutionActorRole::Coco::CertifiedDataEngineer")
if solution_role != "No elements found":
    render_mermaid(solution_role.get("mermaidGraph"))
else:
    print(solution_role)


-----

Peter first task is to set up the data and the data stores that are independent of the hospital supplying the data.  This includes components and templates that are influenced by the shape of the data needed by Coco Pharmaceuticals to validate their new treatment.  Like, Tanya, Peter can begin setting up the data lake and data stores once the data dpecification is agreed since it is used to generate the correct schemas for these stores.

The data specification is also used at a later stage when each hospital is onboarded since the data validator in the onboarding pipeline dynamically validates that the incoming data conforms to the requirements laid out in the data specification.

---

In [31]:
token = peters_client.create_egeria_bearer_token()

project=peters_client.get_element_by_unique_name("Project::" + project_identifier + "::OnboardingPipelines")
if project != "No elements found":
    render_mermaid(project.get("mermaidGraph"))
else:
    print(project)


----

However the first stage is to lay down the hospital independent data stores in the data lake and sandbox used by the data scientist and researchers.

----

In [32]:
token = peters_client.create_egeria_bearer_token()

nominate_hospital_solution_component=peters_client.get_element_by_unique_name("SolutionComponent::Set up Data Lake::V1.0")
if nominate_hospital_solution_component != "No elements found":
    render_mermaid(nominate_hospital_solution_component.get("mermaidGraph"))
else:
    print(nominate_hospital_solution_component)


----

The function below runs the set up data lake for this project.

----

In [33]:
token = peters_client.create_egeria_bearer_token()

airflow_dag_name=set_up_data_lake(peters_client, project_identifier, project_name, project_directory_name, project_schema_name)

print(airflow_dag_name)

ClinicalTrials::PROJ-CT-TBDF::set-up-data-lake [0efe05c7-a880-4fed-a46a-9be7693e8bb8] running
populate_teddy_bear_drop_foot_sandbox


----

One of the services that is configured by "set up data lake" is the integration connector that monitors new files in the Unity Catalog volume that holds the validated files received from the hospitals.  It records the time that the last file was received to help the data scientists to determine if new data has arrived since they last ran their analysis.


In [35]:

display_integration_daemon_status(['MaintainLastUpdateDate', 'FilesCataloguer'], paging=True, width=170)


                                                           Integration Daemon Status @ Wed Jul  1 09:06:01 2026                                                           
╭──────────────────────────────────┬───────────┬─────────────────────────────┬───────────┬────────────────────────────────────────────────────┬──────────────────────────╮
│                                  │           │                             │ Min       │                                                    │                          │
│                                  │           │                             │ Refresh   │                                                    │                          │
│ Connector Name                   │ Status    │ Last Refresh Time           │ (mins)    │ Target Element                                     │ Exception Message        │
├──────────────────────────────────┼───────────┼─────────────────────────────┼───────────┼────────────────────────────────────────────────────┼──

----

It is now possible to see the part of the information supply chain that has just been set up.  It is shown in the "implementation" grey box.  Notice that each component is linked to the solution component it represents.

---

In [36]:
token = peters_client.create_egeria_bearer_token()

information_supply_chain_guid=peters_client.get_element_guid_by_unique_name("InformationSupplyChain::Clinical Trial Treatment Validation Information Supply Chain: " + project_identifier)
if information_supply_chain_guid != "No elements found":
    information_supply_chain=peters_client.get_info_supply_chain_by_guid(guid=information_supply_chain_guid, add_implementation=True)
    render_mermaid(information_supply_chain.get("iscimplementationMermaidGraph"))
else:
    print(information_supply_chain_guid)


-----

The first part of the information supply chain will be added when the hospitals are certified and onboarded into the clinical trial.

-----

### Certifying Hospitals (Round One)

Once a hospital has completed all of the requirements, it can be certified for the clinical trial.  This means it can supply data to the clinical trial.  At the start of the clinical trial, Robbie Record, for Oak Dene and Grant Able for Hampton Hospital are ready to go.  Nellie Dunn still does not have the data sharing agreements signed for Old Market.  As a result, only Oak Dene and Hampton hospitals are certified.

----

In [37]:

tanyas_client = EgeriaTech(tanyas_view_server, tanyas_view_server_url, tanyas_user_id, tanyas_user_pwd)
token = tanyas_client.create_egeria_bearer_token()

oak_dene_certify_process_guid=certify_hospital(tanyas_client, project_identifier, oak_dene_hospital_guid)
hampton_certify_process_guid=certify_hospital(tanyas_client, project_identifier, hampton_hospital_guid)


ClinicalTrials::PROJ-CT-TBDF::certify-hospital [1d91068d-5817-436f-990a-b595154d7343] running for hospital: a3d40103-9e88-4dcb-9c05-82d979c87d3b
ClinicalTrials::PROJ-CT-TBDF::certify-hospital [1d91068d-5817-436f-990a-b595154d7343] running for hospital: 27290e7a-9108-4394-8fdd-cfcee501301a


In [39]:

# Check that each nominate hospital process is "COMPLETED"

token = tanyas_client.create_egeria_bearer_token()

# Show the execution path of the process
process_graph = tanyas_client.get_governance_process_graph(oak_dene_certify_process_guid)
render_mermaid(process_graph.get("governanceActionProcessMermaidGraph"))
process_graph = tanyas_client.get_governance_process_graph(hampton_certify_process_guid)
render_mermaid(process_graph.get("governanceActionProcessMermaidGraph"))



----

## Creating the pipelines for each hospital

The clinical trial is about to start, and so Peter creates the pipelines for the three hospitals, not realizing that Old Market Hospital is not ready.

-----

In [45]:
peters_client = EgeriaTech(peters_view_server, peters_view_server_url, peters_user_id, peters_user_pwd)
token = peters_client.create_egeria_bearer_token()

In [46]:
oak_dene_landing_area_directory_name=onboard_hospital(peters_client, project_identifier, project_directory_name, oak_dene_hospital_guid, oak_dene_landing_folder_directory)
old_market_landing_area_directory_name=onboard_hospital(peters_client, project_identifier, project_directory_name, old_market_hospital_guid, old_market_landing_folder_directory)
hampton_landing_area_directory_name=onboard_hospital(peters_client, project_identifier, project_directory_name, hampton_hospital_guid, hampton_landing_folder_directory)

ClinicalTrials::PROJ-CT-TBDF::onboard-hospital [03b93695-fb31-4e92-8d9f-e2c1073c455c] running for hospital: a3d40103-9e88-4dcb-9c05-82d979c87d3b
ClinicalTrials::PROJ-CT-TBDF::onboard-hospital [03b93695-fb31-4e92-8d9f-e2c1073c455c] running for hospital: 0c899f47-d22f-4d73-86f8-4d30f676acf5
ClinicalTrials::PROJ-CT-TBDF::onboard-hospital [03b93695-fb31-4e92-8d9f-e2c1073c455c] running for hospital: 27290e7a-9108-4394-8fdd-cfcee501301a


----

Notice in **Egeria Operations** that the process for Old Market Hospital fails because it is not certified.

Also notice that the pipelines for Oak Dene and Hampton Hospital are set up ... but nothing for Old Market.  This means that data from uncertified hospitals cannot be accidently included in the clinical trial.

----

In [48]:
display_integration_daemon_status(['MaintainLastUpdateDate', 'FilesCataloguer'], paging=True, width=170)

                                                           Integration Daemon Status @ Wed Jul  1 11:22:08 2026                                                           
╭────────────────────────┬────────┬─────────────────────┬────────┬───────────────────────────────────────────────────────────────────────────────────┬───────────────────╮
│                        │        │                     │ Min    │                                                                                   │                   │
│                        │        │                     │ Refre… │                                                                                   │                   │
│ Connector Name         │ Status │ Last Refresh Time   │ (mins) │ Target Element                                                                    │ Exception Message │
├────────────────────────┼────────┼─────────────────────┼────────┼───────────────────────────────────────────────────────────────────────────────

----

And the lineage now includes the data feeds from these hospitals ...

----

In [49]:

information_supply_chain_guid=peters_client.get_element_guid_by_unique_name("InformationSupplyChain::Clinical Trial Treatment Validation Information Supply Chain: " + project_identifier)
if information_supply_chain_guid != "No elements found":
    information_supply_chain=peters_client.get_info_supply_chain_by_guid(guid=information_supply_chain_guid, add_implementation=True)
    render_mermaid(information_supply_chain.get("iscimplementationMermaidGraph"))
else:
    print(information_supply_chain_guid)


----

### Add data to pipelines

At this point, the processes are set up for Oak Dene Hospital and Hampton Hospital.  It is possible to add the data files to their landing area and they will be systematically added to the data lake folder.   No files are processed for Old Market Hospital.

----

The data onboarding pipeline is called `Onboard Landing Area Files For Teddy Bear Drop Foot Project`.

----

In [52]:

onboarding_pipeline_guid = peters_client.get_guid_for_name(onboarding_pipeline_name) 

process_graph = peters_client.get_governance_process_graph(onboarding_pipeline_guid)
render_mermaid(process_graph.get("governanceActionProcessMermaidGraph"))



---

Let's add Oak Dene Hospital's data for week 1 of the clinical trial ...

---

In [53]:
add_file_to_landing_area(peters_client, oak_dene_source_folder, oak_dene_landing_area_directory_name, "1")

Moving loading-bay/sample-data/oak-dene-drop-foot-weekly-measurements/week1.csv to landing-area/hospitals/oak-dene/clinical-trials/teddy-bear-drop-foot


---

The command below prints out the process steps that have run ...

---

In [58]:

def get_process_instances(egeria_tech, process_name):
    processGUID = egeria_tech.get_element_guid_by_unique_name(process_name)
    if processGUID == "No elements found":
        print(processGUID)
    else:
        processInstances = egeria_tech.get_related_elements(processGUID, "ProcessHierarchy")
        if processInstances:
            if (type(processInstances) == str):
                print(processInstances)
            else:
                print("Process Instances:")
                guids = []
                for processInstance in processInstances:
                    if processInstance:
                       relatedElement = processInstance.get('relatedElement')
                       if relatedElement:
                            guid = relatedElement["elementHeader"]["guid"]
                            guids.append(guid)
                            qualifiedName = relatedElement["properties"]["qualifiedName"]
                            print(f" * {qualifiedName} [{guid}]")
                return guids

def print_process_instances(egeria_client, process_name):
    process_instance_guids = get_process_instances(egeria_client, process_name)
    if process_instance_guids:
        for process_instance_guid in process_instance_guids:
            print(process_instance_guid)
            process_graph = egeria_client.get_gov_action_process_graph(process_instance_guid)
            print_governance_action_process_graph(process_graph)
            

In [59]:
oak_dene_onboarding_process_name="Coco::GovernanceActionProcess::" + project_identifier + "::" + oak_dene_hospital_name + "::WeeklyMeasurements::Onboarding"
print(oak_dene_onboarding_process_name)
print_process_instances(peters_client, oak_dene_onboarding_process_name)

Coco::GovernanceActionProcess::PROJ-CT-TBDF::Oak Dene Hospital::WeeklyMeasurements::Onboarding
No elements found


In [54]:
add_file_to_landing_area(peters_client, hampton_source_folder, hampton_landing_area_directory_name, "1")

Moving loading-bay/sample-data/hampton-drop-foot-weekly-measurements/week1.csv to landing-area/hospitals/hampton/clinical-trials/teddy-bear-drop-foot


In [ ]:
hampton_onboarding_process_name="Coco::GovernanceActionProcess::" + project_identifier + "::" + hampton_hospital_name + "::WeeklyMeasurements::Onboarding"
print(hampton_onboarding_process_name)
print_process_instances(peters_client, hampton_onboarding_process_name)

In [55]:
add_file_to_landing_area(peters_client, oak_dene_source_folder, oak_dene_landing_area_directory_name, "2")
add_file_to_landing_area(peters_client, hampton_source_folder, hampton_landing_area_directory_name, "2")

Moving loading-bay/sample-data/oak-dene-drop-foot-weekly-measurements/week2.csv to landing-area/hospitals/oak-dene/clinical-trials/teddy-bear-drop-foot
Moving loading-bay/sample-data/hampton-drop-foot-weekly-measurements/week2.csv to landing-area/hospitals/hampton/clinical-trials/teddy-bear-drop-foot


In [ ]:
print_process_instances(peters_client, oak_dene_onboarding_process_name)
print_process_instances(peters_client, hampton_onboarding_process_name)

In [56]:
information_supply_chain_guid=peters_client.get_element_guid_by_unique_name("InformationSupplyChain::Clinical Trial Treatment Validation Information Supply Chain: " + project_identifier)
if information_supply_chain_guid != "No elements found":
    information_supply_chain=peters_client.get_info_supply_chain_by_guid(guid=information_supply_chain_guid)
    render_mermaid(information_supply_chain.get("iscimplementationMermaidGraph"))
else:
    print(information_supply_chain_guid)


----

----

In [124]:

solution_component_guid=peters_client.get_element_guid_by_unique_name("SolutionComponent::Treatment Validation Sandbox::V1.0")
if solution_component_guid != "No elements found":
    solution_component=peters_client.get_solution_component_by_guid(guid=solution_component_guid)
    render_mermaid(solution_component.get("mermaidGraph"))
else:
    print(solution_component_guid)


In [125]:

solution_component_guid=peters_client.get_element_guid_by_unique_name("SolutionComponent::Populate Sandbox::V1.0")
if solution_component_guid != "No elements found":
    solution_component=peters_client.get_solution_component_by_guid(guid=solution_component_guid)
    render_mermaid(solution_component.get("mermaidGraph"))
else:
    print(solution_component_guid)


----
### Certifying Old Market Hospital

At last, Old Market Hospital is readdy to join the clinical trial.  Tanya certifies the hospital.

---

In [126]:

tanyas_client = EgeriaTech(tanyas_view_server, tanyas_view_server_url, tanyas_user_id, tanyas_user_pwd)
token = tanyas_client.create_egeria_bearer_token()

certify_hospital(tanyas_client, project_identifier, old_market_hospital_guid)


ClinicalTrials::PROJ-CT-TBDF::certify-hospital [1d91068d-5817-436f-990a-b595154d7343] running for hospital: 0c899f47-d22f-4d73-86f8-4d30f676acf5


'0d133d23-e829-4369-9e5d-6d9bb8bc1f1a'

----

Peter is then able to create the onboarding pipeline for Old Market Hospital

----

In [127]:

peters_client = EgeriaTech(peters_view_server, peters_view_server_url, peters_user_id, peters_user_pwd)
token = peters_client.create_egeria_bearer_token()

old_market_landing_area_directory_name=onboard_hospital(peters_client, project_identifier, project_directory_name, old_market_hospital_guid, old_market_landing_folder_directory)


ClinicalTrials::PROJ-CT-TBDF::onboard-hospital [03b93695-fb31-4e92-8d9f-e2c1073c455c] running for hospital: 0c899f47-d22f-4d73-86f8-4d30f676acf5


In [128]:

display_integration_daemon_status(['MaintainLastUpdateDate', 'FilesCataloguer'], paging=True, width=170)


                                                           Integration Daemon Status @ Wed Jul  1 11:59:07 2026                                                           
╭────────────────────────┬────────┬─────────────────────┬────────┬───────────────────────────────────────────────────────────────────────────────────┬───────────────────╮
│                        │        │                     │ Min    │                                                                                   │                   │
│                        │        │                     │ Refre… │                                                                                   │                   │
│ Connector Name         │ Status │ Last Refresh Time   │ (mins) │ Target Element                                                                    │ Exception Message │
├────────────────────────┼────────┼─────────────────────┼────────┼───────────────────────────────────────────────────────────────────────────────

----

Now any files that are added to Old Market Hospital's landing area are ingested into the data lake. Add week 1 data.

----

In [129]:
add_file_to_landing_area(peters_client, old_market_source_folder, old_market_landing_area_directory_name, "1")

Moving loading-bay/sample-data/old-market-drop-foot-weekly-measurements/week1.csv to landing-area/hospitals/old-market/clinical-trials/teddy-bear-drop-foot


----

Try week 2

-----

In [130]:
add_file_to_landing_area(peters_client, old_market_source_folder, old_market_landing_area_directory_name, "2")

Moving loading-bay/sample-data/old-market-drop-foot-weekly-measurements/week2.csv to landing-area/hospitals/old-market/clinical-trials/teddy-bear-drop-foot


-----

Now Old Market Hospital is caught up and the subsequent weeks can continue in synchronization.

----

In [131]:

add_file_to_landing_area(peters_client, oak_dene_source_folder, oak_dene_landing_area_directory_name, "3")
add_file_to_landing_area(peters_client, hampton_source_folder, hampton_landing_area_directory_name, "3")
add_file_to_landing_area(peters_client, old_market_source_folder, old_market_landing_area_directory_name, "3")


Moving loading-bay/sample-data/oak-dene-drop-foot-weekly-measurements/week3.csv to landing-area/hospitals/oak-dene/clinical-trials/teddy-bear-drop-foot
Moving loading-bay/sample-data/hampton-drop-foot-weekly-measurements/week3.csv to landing-area/hospitals/hampton/clinical-trials/teddy-bear-drop-foot
Moving loading-bay/sample-data/old-market-drop-foot-weekly-measurements/week3.csv to landing-area/hospitals/old-market/clinical-trials/teddy-bear-drop-foot


In [132]:

add_file_to_landing_area(peters_client, oak_dene_source_folder, oak_dene_landing_area_directory_name, "4")
add_file_to_landing_area(peters_client, hampton_source_folder, hampton_landing_area_directory_name, "4")
add_file_to_landing_area(peters_client, old_market_source_folder, old_market_landing_area_directory_name, "4")


Moving loading-bay/sample-data/oak-dene-drop-foot-weekly-measurements/week4.csv to landing-area/hospitals/oak-dene/clinical-trials/teddy-bear-drop-foot
Moving loading-bay/sample-data/hampton-drop-foot-weekly-measurements/week4.csv to landing-area/hospitals/hampton/clinical-trials/teddy-bear-drop-foot
Moving loading-bay/sample-data/old-market-drop-foot-weekly-measurements/week4.csv to landing-area/hospitals/old-market/clinical-trials/teddy-bear-drop-foot


In [133]:

add_file_to_landing_area(peters_client, oak_dene_source_folder, oak_dene_landing_area_directory_name, "5")
add_file_to_landing_area(peters_client, hampton_source_folder, hampton_landing_area_directory_name, "5")
add_file_to_landing_area(peters_client, old_market_source_folder, old_market_landing_area_directory_name, "5")


Moving loading-bay/sample-data/oak-dene-drop-foot-weekly-measurements/week5.csv to landing-area/hospitals/oak-dene/clinical-trials/teddy-bear-drop-foot
Moving loading-bay/sample-data/hampton-drop-foot-weekly-measurements/week5.csv to landing-area/hospitals/hampton/clinical-trials/teddy-bear-drop-foot
Moving loading-bay/sample-data/old-market-drop-foot-weekly-measurements/week5.csv to landing-area/hospitals/old-market/clinical-trials/teddy-bear-drop-foot


----
### After all hospitals are registered ...

---

In [134]:

callies_client = EgeriaTech(callies_view_server, callies_view_server_url, callies_user_id, callies_user_pwd)
token = callies_client.create_egeria_bearer_token()


In [135]:

information_supply_chain_guid=callies_client.get_element_guid_by_unique_name("InformationSupplyChain::Clinical Trial Treatment Validation Information Supply Chain: " + project_identifier)
if information_supply_chain_guid != "No elements found":
    information_supply_chain=callies_client.get_info_supply_chain_by_guid(guid=information_supply_chain_guid, add_implementation=True, body={"class" : "GetRequestBody", "forLineage" : True })
    render_mermaid(information_supply_chain.get("iscimplementationMermaidGraph"))
else:
    print(information_supply_chain_guid)


------

## Dragon Breath Clinical Trial

The cells below step through setting up a new clinical trial using the same steps used for Teddy Bear Dop Foot to show the use of both the standard and specialized resources for a clinical trial.


----

In [143]:

print()

project_identifier="PROJ-CT-DB"
project_name="Dragon Breath Clinical Trial"
project_description="Clinical trial related to the new treatment to enhance the volume and quality of the lungs."
project_directory_name="dragon-breath"
project_schema_name="dragon_breath"

print("Project " + project_identifier + ": " + project_name)
print("  " + project_description)



Project PROJ-CT-DB: Dragon Breath Clinical Trial
  Clinical trial related to the new treatment to enhance the volume and quality of the lungs.


In [144]:

tessas_client = EgeriaTech(tessas_view_server, tessas_view_server_url, tessas_user_id, tessas_user_pwd)
token = tessas_client.create_egeria_bearer_token()

set_up_clinical_trial(tessas_client,
                      parent_project_guid,
                      clinical_trial_owner_guid,
                      clinical_trial_manager_guid,
                      project_manager_guid,
                      data_engineer_guid,
                      integration_developer_guid,
                      data_scientist_guid,
                      project_identifier, 
                      project_name,
                      project_description) 


'38814fb4-77e7-46f4-a4eb-b949ffd9d8bd'

In [147]:

tanyas_client = EgeriaTech(tanyas_view_server, tanyas_view_server_url, tanyas_user_id, tanyas_user_pwd)
token = tanyas_client.create_egeria_bearer_token()

nominate_hospital(tanyas_client, project_identifier, bowden_arrow_hospital_guid, bowden_arrow_contact_person_guid)


ClinicalTrials::PROJ-CT-DB::nominate-hospital [89724f2b-750e-484c-aee3-2ea592ed054e] running for hospital: 6bdd3666-7d46-4427-9316-1df0dce8d6cc


'9ae01299-35df-44d0-86ef-f1da05f6173f'

In [148]:

peters_client = EgeriaTech(peters_view_server, peters_view_server_url, peters_user_id, peters_user_pwd)
token = peters_client.create_egeria_bearer_token()

set_up_data_lake(peters_client, project_identifier, project_name, project_directory_name, project_schema_name)


ClinicalTrials::PROJ-CT-DB::set-up-data-lake [8f83c65b-3b5e-4627-bacd-7a6016c0a8d2] running


'populate_dragon_breath_sandbox'

In [149]:

tanyas_client = EgeriaTech(tanyas_view_server, tanyas_view_server_url, tanyas_user_id, tanyas_user_pwd)
token = tanyas_client.create_egeria_bearer_token()

certify_hospital(tanyas_client, project_identifier, bowden_arrow_hospital_guid)


ClinicalTrials::PROJ-CT-DB::certify-hospital [f096ed24-713d-4e0a-83b9-488138c826e3] running for hospital: 6bdd3666-7d46-4427-9316-1df0dce8d6cc


'b9730e73-2daf-4cf7-9421-d970530136e0'

In [150]:
peters_client = EgeriaTech(peters_view_server, peters_view_server_url, peters_user_id, peters_user_pwd)
token = peters_client.create_egeria_bearer_token()

bowden_arrow_landing_area_directory_name=onboard_hospital(peters_client, project_identifier, project_directory_name, bowden_arrow_hospital_guid, bowden_arrow_landing_folder_directory)

ClinicalTrials::PROJ-CT-DB::onboard-hospital [c5eabf1a-6ca0-4c1e-9b52-9573f13f2f67] running for hospital: 6bdd3666-7d46-4427-9316-1df0dce8d6cc


In [151]:

information_supply_chain_guid=peters_client.get_element_guid_by_unique_name("InformationSupplyChain::Clinical Trial Treatment Validation Information Supply Chain: " + project_identifier)
if information_supply_chain_guid != "No elements found":
    information_supply_chain=peters_client.get_info_supply_chain_by_guid(guid=information_supply_chain_guid, add_implementation=True, body={"class" : "GetRequestBody", "forLineage" : True })
    render_mermaid(information_supply_chain.get("iscimplementationMermaidGraph"))
else:
    print(information_supply_chain_guid)
    

-----

## Falcon Feather Mite


In [152]:

print()

project_identifier="PROJ-CT-FFM"
project_name="Falcon Feather Mite Clinical Trial"
project_description="Clinical trial related to the new treatment to cuew feather mites in falcons."
project_directory_name="falcon-feather-mite"
project_schema_name="falcon-feather-mite"

print("Project " + project_identifier + ": " + project_name)
print("  " + project_description)



Project PROJ-CT-FFM: Falcon Feather Mite Clinical Trial
  Clinical trial related to the new treatment to cuew feather mites in falcons.


In [153]:

tessas_client = EgeriaTech(tessas_view_server, tessas_view_server_url, tessas_user_id, tessas_user_pwd)
token = tessas_client.create_egeria_bearer_token()

set_up_clinical_trial(tessas_client,
                      parent_project_guid,
                      clinical_trial_owner_guid,
                      clinical_trial_manager_guid,
                      project_manager_guid,
                      data_engineer_guid,
                      integration_developer_guid,
                      data_scientist_guid,
                      project_identifier, 
                      project_name,
                      project_description) 


'd0a22ed5-849a-4c5c-a727-796f0bc3d978'

In [157]:

tanyas_client = EgeriaTech(tanyas_view_server, tanyas_view_server_url, tanyas_user_id, tanyas_user_pwd)
token = tanyas_client.create_egeria_bearer_token()

nominate_hospital(tanyas_client, project_identifier, old_market_hospital_guid, old_market_contact_person_guid)
nominate_hospital(tanyas_client, project_identifier, hampton_hospital_guid, hampton_contact_person_guid)


ClinicalTrials::PROJ-CT-FFM::nominate-hospital [a9702560-8214-4609-bcbe-a8c709f90469] running for hospital: 0c899f47-d22f-4d73-86f8-4d30f676acf5
ClinicalTrials::PROJ-CT-FFM::nominate-hospital [a9702560-8214-4609-bcbe-a8c709f90469] running for hospital: 27290e7a-9108-4394-8fdd-cfcee501301a


'd0881967-71e4-4273-a404-4c59eb4983da'

In [158]:

peters_client = EgeriaTech(peters_view_server, peters_view_server_url, peters_user_id, peters_user_pwd)
token = peters_client.create_egeria_bearer_token()

set_up_data_lake(peters_client, project_identifier, project_name, project_directory_name, project_schema_name)


ClinicalTrials::PROJ-CT-FFM::set-up-data-lake [33a1517f-da4b-42b2-aab5-a2cb8d2a8221] running


'populate_falcon-feather-mite_sandbox'

In [159]:

tanyas_client = EgeriaTech(tanyas_view_server, tanyas_view_server_url, tanyas_user_id, tanyas_user_pwd)
token = tanyas_client.create_egeria_bearer_token()

certify_hospital(tanyas_client, project_identifier, old_market_hospital_guid)
certify_hospital(tanyas_client, project_identifier, hampton_hospital_guid)


ClinicalTrials::PROJ-CT-FFM::certify-hospital [e88556c8-19a2-403b-8bc6-168b97de18c6] running for hospital: 0c899f47-d22f-4d73-86f8-4d30f676acf5
ClinicalTrials::PROJ-CT-FFM::certify-hospital [e88556c8-19a2-403b-8bc6-168b97de18c6] running for hospital: 27290e7a-9108-4394-8fdd-cfcee501301a


'0cc7c688-0cc6-45b5-883c-327683857ef0'

In [160]:
peters_client = EgeriaTech(peters_view_server, peters_view_server_url, peters_user_id, peters_user_pwd)
token = peters_client.create_egeria_bearer_token()

old_market_landing_area_directory_name=onboard_hospital(peters_client, project_identifier, project_directory_name, old_market_hospital_guid, old_market_landing_folder_directory)
hampton_landing_area_directory_name=onboard_hospital(peters_client, project_identifier, project_directory_name, hampton_hospital_guid, hampton_landing_folder_directory)

ClinicalTrials::PROJ-CT-FFM::onboard-hospital [846dc79c-9323-4a9e-be0c-9dd7ef4355cb] running for hospital: 0c899f47-d22f-4d73-86f8-4d30f676acf5
ClinicalTrials::PROJ-CT-FFM::onboard-hospital [846dc79c-9323-4a9e-be0c-9dd7ef4355cb] running for hospital: 27290e7a-9108-4394-8fdd-cfcee501301a


In [167]:

information_supply_chain_guid=peters_client.get_element_guid_by_unique_name("InformationSupplyChain::Clinical Trial Treatment Validation Information Supply Chain: " + project_identifier)
if information_supply_chain_guid != "No elements found":
    information_supply_chain=peters_client.get_info_supply_chain_by_guid(guid=information_supply_chain_guid, add_implementation=True, body={"class" : "GetRequestBody", "forLineage" : True })
    render_mermaid(information_supply_chain.get("iscimplementationMermaidGraph"))
else:
    print(information_supply_chain_guid)
    

-----

## Werewolf Transformation


In [162]:

print()

project_identifier="PROJ-CT-WTF"
project_name="Werewolf Transformation Clinical Trial"
project_description="Clinical trial related to the new treatment to minimise or prevent werewolf transformation."
project_directory_name="werewolf-transformation"
project_schema_name="werewolf_transformation"

print("Project " + project_identifier + ": " + project_name)
print("  " + project_description)



Project PROJ-CT-WTF: Werewolf Transformation Clinical Trial
  Clinical trial related to the new treatment to minimise or prevent werewolf transformation.


In [163]:

tessas_client = EgeriaTech(tessas_view_server, tessas_view_server_url, tessas_user_id, tessas_user_pwd)
token = tessas_client.create_egeria_bearer_token()

set_up_clinical_trial(tessas_client,
                      parent_project_guid,
                      clinical_trial_owner_guid,
                      clinical_trial_manager_guid,
                      project_manager_guid,
                      data_engineer_guid,
                      integration_developer_guid,
                      data_scientist_guid,
                      project_identifier, 
                      project_name,
                      project_description) 


'3f337aa4-01f7-4414-ab2d-9ccf1bbccf59'

In [164]:

tanyas_client = EgeriaTech(tanyas_view_server, tanyas_view_server_url, tanyas_user_id, tanyas_user_pwd)
token = tanyas_client.create_egeria_bearer_token()

nominate_hospital(tanyas_client, project_identifier, oak_dene_hospital_guid, oak_dene_contact_person_guid)
nominate_hospital(tanyas_client, project_identifier, old_market_hospital_guid, old_market_contact_person_guid)
nominate_hospital(tanyas_client, project_identifier, hampton_hospital_guid, hampton_contact_person_guid)
nominate_hospital(tanyas_client, project_identifier, bowden_arrow_hospital_guid, bowden_arrow_contact_person_guid)


ClinicalTrials::PROJ-CT-WTF::nominate-hospital [ec1f3145-7e9f-4f7c-a21d-d07ff964f689] running for hospital: a3d40103-9e88-4dcb-9c05-82d979c87d3b
ClinicalTrials::PROJ-CT-WTF::nominate-hospital [ec1f3145-7e9f-4f7c-a21d-d07ff964f689] running for hospital: 0c899f47-d22f-4d73-86f8-4d30f676acf5
ClinicalTrials::PROJ-CT-WTF::nominate-hospital [ec1f3145-7e9f-4f7c-a21d-d07ff964f689] running for hospital: 27290e7a-9108-4394-8fdd-cfcee501301a
ClinicalTrials::PROJ-CT-WTF::nominate-hospital [ec1f3145-7e9f-4f7c-a21d-d07ff964f689] running for hospital: 6bdd3666-7d46-4427-9316-1df0dce8d6cc


'f75a5546-712a-4191-ab5f-b5c3e2f8d2c0'

In [165]:

peters_client = EgeriaTech(peters_view_server, peters_view_server_url, peters_user_id, peters_user_pwd)
token = peters_client.create_egeria_bearer_token()

set_up_data_lake(peters_client, project_identifier, project_name, project_directory_name, project_schema_name)


ClinicalTrials::PROJ-CT-WTF::set-up-data-lake [bcf5b609-b3d7-42c3-8b13-5ce117716f26] running


'populate_werewolf_transformation_sandbox'

In [166]:

tanyas_client = EgeriaTech(tanyas_view_server, tanyas_view_server_url, tanyas_user_id, tanyas_user_pwd)
token = tanyas_client.create_egeria_bearer_token()

certify_hospital(tanyas_client, project_identifier, oak_dene_hospital_guid)
certify_hospital(tanyas_client, project_identifier, old_market_hospital_guid)
certify_hospital(tanyas_client, project_identifier, hampton_hospital_guid)
certify_hospital(tanyas_client, project_identifier, bowden_arrow_hospital_guid)


ClinicalTrials::PROJ-CT-WTF::certify-hospital [8af5901a-cb86-41c1-8821-d2b3be0fc4f9] running for hospital: a3d40103-9e88-4dcb-9c05-82d979c87d3b
ClinicalTrials::PROJ-CT-WTF::certify-hospital [8af5901a-cb86-41c1-8821-d2b3be0fc4f9] running for hospital: 0c899f47-d22f-4d73-86f8-4d30f676acf5
ClinicalTrials::PROJ-CT-WTF::certify-hospital [8af5901a-cb86-41c1-8821-d2b3be0fc4f9] running for hospital: 27290e7a-9108-4394-8fdd-cfcee501301a
ClinicalTrials::PROJ-CT-WTF::certify-hospital [8af5901a-cb86-41c1-8821-d2b3be0fc4f9] running for hospital: 6bdd3666-7d46-4427-9316-1df0dce8d6cc


'75ecfb03-a43a-4d22-a928-66685eb4a03f'

In [168]:
peters_client = EgeriaTech(peters_view_server, peters_view_server_url, peters_user_id, peters_user_pwd)
token = peters_client.create_egeria_bearer_token()

oak_dene_landing_area_directory_name=onboard_hospital(peters_client, project_identifier, project_directory_name, oak_dene_hospital_guid, oak_dene_landing_folder_directory)
old_market_landing_area_directory_name=onboard_hospital(peters_client, project_identifier, project_directory_name, old_market_hospital_guid, old_market_landing_folder_directory)
hampton_landing_area_directory_name=onboard_hospital(peters_client, project_identifier, project_directory_name, hampton_hospital_guid, hampton_landing_folder_directory)
bowden_arrow_landing_area_directory_name=onboard_hospital(peters_client, project_identifier, project_directory_name, bowden_arrow_hospital_guid, bowden_arrow_landing_folder_directory)

ClinicalTrials::PROJ-CT-WTF::onboard-hospital [da5f7444-48be-476f-8731-4a5b68d93988] running for hospital: a3d40103-9e88-4dcb-9c05-82d979c87d3b
ClinicalTrials::PROJ-CT-WTF::onboard-hospital [da5f7444-48be-476f-8731-4a5b68d93988] running for hospital: 0c899f47-d22f-4d73-86f8-4d30f676acf5
ClinicalTrials::PROJ-CT-WTF::onboard-hospital [da5f7444-48be-476f-8731-4a5b68d93988] running for hospital: 27290e7a-9108-4394-8fdd-cfcee501301a
ClinicalTrials::PROJ-CT-WTF::onboard-hospital [da5f7444-48be-476f-8731-4a5b68d93988] running for hospital: 6bdd3666-7d46-4427-9316-1df0dce8d6cc


In [170]:


information_supply_chain_guid=peters_client.get_element_guid_by_unique_name("InformationSupplyChain::Clinical Trial Treatment Validation Information Supply Chain: " + project_identifier)
if information_supply_chain_guid != "No elements found":
    information_supply_chain=peters_client.get_info_supply_chain_by_guid(guid=information_supply_chain_guid, add_implementation=True, body={"class" : "GetRequestBody", "forLineage" : True })
    render_mermaid(information_supply_chain.get("iscimplementationMermaidGraph"))
else:
    print(information_supply_chain_guid)
    


In [172]:

add_file_to_landing_area(peters_client, oak_dene_source_folder, oak_dene_landing_area_directory_name, "1")


Moving loading-bay/sample-data/oak-dene-drop-foot-weekly-measurements/week1.csv to landing-area/hospitals/oak-dene/clinical-trials/werewolf-transformation
